# G4 - cross-lineage and cross-architecture transfer, calibrated

Both held-out encoders through one pipeline, with SigLIP as the control. SigLIP published 94.2%, rebuilt 86.4%, ConvNeXt 87.5% - measured identically, a convnet sits ABOVE an encoder known to be in-band, so **architecture does not bound the claim**. Row correspondence is tested, not assumed.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G4 - cross-lineage AND cross-architecture transfer, calibrated.
# Supersedes the separate ConvNeXt notebook: this loop measures both
# held-out encoders and the SigLIP control in one pass.
#
# THE PROBLEM WITH READING 88.5% ON ITS OWN. The 93.8-96.5% band and
# SigLIP's 94.2% were produced by the ORIGINAL head protocol. No head
# artifact survived, so tonight's head was refitted from scratch: 512-d hub
# coordinates, SBERT targets, ridge at alpha=1.0. Comparing a number from
# the new pipeline against a band from the old one assumes the two agree,
# and nothing has checked that.
#
# Two reasons to doubt it. The source encoder scored 0.433 against the
# report's 0.398 / 0.442 / 0.468 - the right regime, not the same numbers.
# And ConvNeXt's NATIVE R@1 (0.478) came out above img_small's own (0.433),
# so the denominator in that ratio is a stronger ceiling than the source.
#
# THE CONTROL. SigLIP 2 is cached, covers the identical 9,533 COCO ids as
# ConvNeXt, and has a PUBLISHED value of 94.2% from the original protocol.
# Push it through tonight's pipeline and the offset becomes measurable.
#
# PRE-REGISTERED READING, fixed before running:
#   SigLIP rebuilt lands within ~2 points of 94.2%
#       -> the protocol is calibrated. ConvNeXt's shortfall is REAL and
#          architecture (or objective) costs something measurable.
#   SigLIP rebuilt lands near 88%, like ConvNeXt
#       -> the offset is the PROTOCOL, not the encoder. ConvNeXt is
#          effectively in-band and the honest report is "no encoder tested
#          falls outside the band", with the rebuilt protocol noted.
#   SigLIP rebuilt lands somewhere else entirely
#       -> the reconstruction does not reproduce G4 and NEITHER number is
#          quotable. Report both as not run.
#
# This is the difference between a finding and an artifact, and it costs
# one extra ridge fit.
# ==========================================================
import os
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ.get("DATA_DIR", "."))

HUB_NPZ = DATA_DIR / "hub_rebuilt.npz"    # written by G0 - run G0 first
# Everything else comes out of that file: the aligned raw_* spaces, the hub
# coordinates, and the split. Only ConvNeXt is loaded separately.
# ----------------------------------------------------------------------

N_TRAIN, N_EVAL = 8533, 1000
BAND = (0.938, 0.965)   # within-family zero-shot band from C.13.2
SEED = 0

# The two held-out encoders. SigLIP carries its PUBLISHED value from the
# original protocol; ConvNeXt has none, which is the whole point - SigLIP
# calibrates the rebuilt pipeline, ConvNeXt is measured through it.
HELDOUT = {
    "SigLIP 2": ("e1_img_ckpt_siglip2-base-patch16-224_native.npz", 0.942),
    "ConvNeXt": ("e1_img_ckpt_convnext-base-224-22k_native.npz", None),
}

hub = np.load(HUB_NPZ, allow_pickle=True)
H_all = hub["H_all"]
N_TR, N_EV = int(hub["n_train"]), int(hub["n_eval"])
HUB_WIDTH = 512
H = H_all[:, :HUB_WIDTH]

T = hub["raw_txt_sbert"].astype(np.float64)
T = T / (np.linalg.norm(T, axis=1, keepdims=True) + 1e-8)
X_ref = hub["raw_img_small"].astype(np.float64)

from sklearn.linear_model import Ridge


def r1(P, G):
    P = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-8)
    return float(((P @ G.T).argmax(1) == np.arange(len(P))).mean())


def r2(src, tgt):
    W = Ridge(alpha=1.0).fit(src[:N_TR], tgt[:N_TR]).coef_.T
    p, t = src[N_TR:] @ W, tgt[N_TR:]
    return 1.0 - float(((t - p) ** 2).sum() / (t ** 2).sum())


W_ref = Ridge(alpha=1.0).fit(X_ref[:N_TR], H[:N_TR]).coef_.T
H_ref = X_ref @ W_ref
W_head = Ridge(alpha=1.0).fit(H_ref[:N_TR], T[:N_TR]).coef_.T
r_source = r1(H_ref[N_TR:] @ W_head, T[N_TR:])
print(f"source encoder img_small R@1 = {r_source:.3f}   "
      f"(report range 0.398 / 0.442 / 0.468)")

rng = np.random.default_rng(SEED)
rows = []
for label, (fn, published) in HELDOUT.items():
    z = np.load(DATA_DIR / fn, allow_pickle=True)
    Xh = z["img"].astype(np.float64)
    Xh = Xh / (np.linalg.norm(Xh, axis=1, keepdims=True) + 1e-8)
    assert len(Xh) == len(H_all), f"{label}: row count differs"

    # Row correspondence is TESTED, not assumed. The DINOv2 caches store
    # keep = arange (positional), the held-out caches store real COCO ids,
    # so an id join is impossible and positional identity is the only
    # available hypothesis. A misaligned encoder yields a LOW transfer
    # number indistinguishable from a real architecture effect, which is
    # the most dangerous failure mode in this test.
    sep = r2(X_ref, Xh) - r2(X_ref, Xh[rng.permutation(len(Xh))])
    assert sep > 0.05, (
        f"{label}: rows do not correspond (separation {sep:.3f}). "
        "Do not read any transfer number from this encoder.")

    # collapse diagnostic on the held-out space itself
    _sub = Xh[rng.choice(len(Xh), 1500, replace=False)]
    _pc = float((_sub @ _sub.T)[np.triu_indices(1500, 1)].mean())
    W_h = Ridge(alpha=1.0).fit(Xh[:N_TR], H[:N_TR]).coef_.T
    Hh = Xh @ W_h
    zero = r1(Hh[N_TR:] @ W_head, T[N_TR:])
    W_nat = Ridge(alpha=1.0).fit(Hh[:N_TR], T[:N_TR]).coef_.T
    nat = r1(Hh[N_TR:] @ W_nat, T[N_TR:])
    Wr = rng.normal(size=W_h.shape) / np.sqrt(W_h.shape[0])
    ctrl = r1((Xh[N_TR:] @ Wr) @ W_head, T[N_TR:])
    rows.append((label, published, zero, nat, zero / nat, ctrl, sep))
    print(f"  {label}: width {Xh.shape[1]}, "
          f"{N_TR / Xh.shape[1]:.1f} rows/dim, pair-cos {_pc:+.3f}")

print("\n" + "=" * 78)
print(f"{'encoder':<12}{'published':>11}{'rebuilt':>10}{'zero':>8}"
      f"{'native':>9}{'control':>9}{'align sep':>11}")
print("=" * 78)
for lb, pub, z_, n_, pct, ct, sp in rows:
    ps = f"{pub:.1%}" if pub else "-"
    print(f"{lb:<12}{ps:>11}{pct:>10.1%}{z_:>8.3f}{n_:>9.3f}"
          f"{ct:>9.3f}{sp:>11.3f}")

sig = next((r for r in rows if r[1]), None)
cnv = next((r for r in rows if not r[1]), None)
print("\n" + "=" * 78)
if sig:
    off = sig[4] - sig[1]
    print(f"SigLIP offset vs published: {off:+.1%}")
    if abs(off) <= 0.02:
        print("PROTOCOL CALIBRATED. The rebuilt pipeline reproduces SigLIP's")
        print("published value, so ConvNeXt's shortfall is a real effect.")
        print(f"Report ConvNeXt at {cnv[4]:.1%}, below the 93.8-96.5% band,")
        print("and state that architecture and objective are confounded.")
    elif abs(cnv[4] - sig[4]) < 0.02:
        print("PROTOCOL OFFSET. SigLIP lands near ConvNeXt under the rebuilt")
        print("head, so the gap is the protocol and not the encoder. Corrected")
        print(f"for the offset, ConvNeXt sits at about {cnv[4] - off:.1%} -")
        print("inside the band. Report as 'no encoder tested falls outside',")
        print("and say the protocol was reconstructed.")
    else:
        print("NEITHER. The rebuilt pipeline does not reproduce SigLIP and")
        print("ConvNeXt does not track it either. The reconstruction is not")
        print("G4's protocol. Report both as not run - a number you cannot")
        print("calibrate is worse than a gap you can name.")


print("\nScope to state with whichever verdict lands: one convnet, one")
print("image domain, ImageNet-22k SUPERVISED rather than self-supervised.")
print("ConvNeXt differs from DINOv2 in architecture AND objective, so a")
print("shortfall cannot be attributed to architecture alone. Since C.13.5")
print("ranks objective above lineage above modality, objective is the more")
print("likely of the two. The honest framing is 'no encoder tested falls")
print("outside the band', not 'architecture is irrelevant'.")